## 1 - Configuração

In [0]:
import json
from pyspark.sql import functions as F

def parametro(nome, padrao):
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao

catalog = parametro("catalog", "case")
gold_schema = f"`{catalog}`.gold"

In [0]:
ft_chamados = spark.table(f"{gold_schema}.ft_chamados")
dim_atendente = spark.table(f"{gold_schema}.dim_atendente")

display(ft_chamados.limit(10))
display(dim_atendente.limit(10))

## 2 - Join 
Nível Consolidado: Mapeia ura e chatbot como Autosservico. Garante N1 para atendimento inicial e N2 para atendimento especializado, preservando o nível do canal mesmo quando id_atendente = -1 (abandonos).

Flag de Resolução: Converte resolvido = 'sim' para 1 e demais casos para 0.

Tratamento de Custo: Aplica coalesce convertendo nulos para 0.0 contábil.

In [0]:
df_base = ft_chamados.alias("f").join(
    dim_atendente.alias("a"),
    on="id_atendente",
    how="left"
).withColumn(
    "ano_mes", F.col("f.mes_abertura")
).withColumn(
    "nivel_consolidado",
    F.when(F.col("f.canal").isin("ura", "chatbot"), F.lit("Autosservico"))
     .when(F.col("f.canal") == "atendimento inicial", F.lit("N1"))
     .when(F.col("f.canal") == "atendimento especializado", F.lit("N2"))
     .when(F.col("a.nivel_atendimento").isNotNull(), F.col("a.nivel_atendimento"))
     .otherwise(F.lit("Nao Atribuido"))
).withColumn(
    "is_resolvido", F.when(F.col("f.resolvido") == "sim", 1).otherwise(0)
).withColumn(
    "valor_custo", F.coalesce(F.col("f.custo"), F.lit(0.0))
)

## 3 Agregação da Visão macro e formulas
taxa_resolucao: (Total de Resolvidos / Total de Chamados) * 100

custo_medio_chamado: Custo Total / Total de Chamados

custo_por_resolvido: Custo Total / Total de Chamados Resolvidos

espera_media_seg: Média Aritmética do Tempo em Fila (Retorna 0.0s para URA/Chatbot)

atendimento_med_seg: Média Aritmética da Duração Ativa da Chamada

qtd_atendentes_ativos: Contagem Distinta de id_atendente (!= -1) apenas nos canais humanos (N1/N2)

chamados_por_atendente: Total de Chamados / Total de Atendentes Ativos no Canal/Mês
     Proteção: Se qtd_atendentes_ativos == 0 -> Retorna NULL (evita divisão por zero no Autosserviço)

In [0]:
kpi_custos_produtividade = df_base.filter(
    F.col("ano_mes").isNotNull()
).groupBy(
    F.col("ano_mes").alias("mes"),
    F.col("f.canal").alias("canal"),
    F.col("nivel_consolidado").alias("nivel")
).agg(
    F.countDistinct("f.id_chamado").alias("qtd_chamados"),
    F.sum("is_resolvido").alias("qtd_resolvidos"),
    F.round((F.sum("is_resolvido") / F.countDistinct("f.id_chamado")) * 100, 2).alias("taxa_resolucao"),
    
    
    F.round(F.sum("valor_custo"), 2).alias("custo_total"),
    F.round(F.sum("valor_custo") / F.countDistinct("f.id_chamado"), 2).alias("custo_medio_chamado"),
    F.round(
        F.sum("valor_custo") / F.when(F.sum("is_resolvido") == 0, F.lit(None)).otherwise(F.sum("is_resolvido")), 2
    ).alias("custo_por_resolvido"),
    F.round(F.avg("f.tempo_espera"), 1).alias("espera_media_seg"),
    F.round(F.avg("f.tempo_atendimento"), 1).alias("atendimento_medio_seg"),
    F.countDistinct(
        F.when(
            (F.col("f.id_atendente") != -1) & (~F.col("f.canal").isin("ura", "chatbot")), 
            F.col("f.id_atendente")
        )
    ).alias("qtd_atendentes_ativos")
).withColumn(
    "chamados_por_atendente",
    F.round(
        F.col("qtd_chamados") / F.when(F.col("qtd_atendentes_ativos") == 0, F.lit(None)).otherwise(F.col("qtd_atendentes_ativos")), 1
    )
).orderBy("mes", "canal", "nivel")

## 4- Agregação da Visao micro e formulas
qtd_atendimentos: Volume total de atendimentos individuais no mes

taxa_resolucao: (Resolvidos pelo Atendente / Total de Atendimentos do Atendente) * 100

tempo_medio: Média Aritmética do Tempo de Atendimento do Operador

custo_total: Soma dos Custos dos Chamados Atendidos pelo Operador

In [0]:
kpi_produtividade_atendente = df_base.filter(
    (F.col("f.id_atendente") != -1) & F.col("ano_mes").isNotNull()
).groupBy(
    F.col("ano_mes").alias("mes"),
    F.col("f.id_atendente").alias("id_atendente"),
    F.col("a.nome_atendente").alias("nome_atendente"),
    F.col("a.nivel_atendimento").alias("nivel_atendimento")
).agg(
    F.countDistinct("f.id_chamado").alias("qtd_atendimentos"),
    F.sum("is_resolvido").alias("qtd_resolvidos"),
    F.round((F.sum("is_resolvido") / F.countDistinct("f.id_chamado")) * 100, 2).alias("taxa_resolucao"),
    F.round(F.avg("f.tempo_atendimento"), 1).alias("tempo_medio"),
    F.round(F.sum("valor_custo"), 2).alias("custo_total")
).orderBy("mes", "id_atendente")

In [0]:
display(kpi_custos_produtividade.limit(10))
display(kpi_produtividade_atendente.limit(10))




## 5 - Publicação delta e Exportação Csv

Grava as tabelas Delta de forma idempotente (overwrite + overwriteSchema) e gera os arquivos CSV compactados em partição única (coalesce(1)) na pasta de exportação de Volumes do Databricks para leitura no Power BI.

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.gold.export")
kpi_custos_produtividade.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.kpi_custos_produtividade")
kpi_produtividade_atendente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.kpi_produtividade_atendente")


export_dir_macro = "/Volumes/case/gold/export/kpi_custos_produtividade"
export_dir_micro = "/Volumes/case/gold/export/kpi_produtividade_atendente"

kpi_custos_produtividade.coalesce(1).write.mode("overwrite").option("header", "true").csv(export_dir_macro)
kpi_produtividade_atendente.coalesce(1).write.mode("overwrite").option("header", "true").csv(export_dir_micro)


##6 - Portão de integridade e Qualidade

Executa uma verificação na memória que interrompe a execução e aciona erro no Job se a contagem de linhas das tabelas agregadas for zerada (count() == 0).

In [0]:
assert kpi_custos_produtividade.count() > 0, "Erro: kpi_custos_produtividade gerou 0 linhas!"
assert kpi_produtividade_atendente.count() > 0, "Erro: kpi_produtividade_atendente gerou 0 linhas!"

print("=== Processamento concluído com sucesso! ===")